# Normalizacao do relatorio Sem Parar

Objetivo:
- Ler `02-Referencias/relatorio_20260331024626.xlsx`
- Padronizar colunas entre abas
- Gerar dataframes consolidados para analise

Este notebook foi pensado para arquivos analogos as faturas em PDF (`FATURA GEG.pdf` e `FATURA RSE.pdf`).

In [2]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

ARQUIVO = Path("../02-Referencias/relatorio_20260331024626.xlsx")
if not ARQUIVO.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {ARQUIVO.resolve()}")

xls = pd.ExcelFile(ARQUIVO)
xls.sheet_names

['RESUMO DA FATURA',
 'RESUMO PASSAGENS PEDÁGIO',
 'RESUMO PASSAGENS ESTACIONAMENTO',
 'RESUMO TRANS ESTABELECIMENTO',
 'PASSAGENS PEDÁGIO',
 'PASSAGENS ESTACIONAMENTO',
 'PASSAGENS VALE-PEDÁGIO',
 'TRANSAÇÕES ESTABELECIMENTOS',
 'ADESÕES',
 'MENSALIDADES',
 'OUTROS SERVICOS',
 'CRÉDITOS',
 'DÉBITOS']

In [8]:
def normalizar_texto(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")

def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalizar_texto(c) for c in out.columns]
    return out

def parse_valor(x):
    if pd.isna(x):
        return pd.NA
    if isinstance(x, (int, float)):
        return float(x)
    t = str(x).strip().replace(".", "").replace(",", ".")
    t = re.sub(r"[^0-9\.-]", "", t)
    if t in {"", "-", "."}:
        return pd.NA
    try:
        return float(t)
    except Exception:
        return pd.NA

MAPA_TIPO = {
    "passagens_pedagio": "pedagio",
    "passagens_estacionamento": "estacionamento",
    "passagens_vale_pedagio": "vale_pedagio",
    "transacoes_estabelecimentos": "estabelecimento",
    "adesoes": "adesao",
    "mensalidades": "mensalidade",
    "outros_servicos": "outro_servico",
    "creditos": "credito",
    "debitos": "debito",
}

MAPA_COLUNAS = {
    "placa": "placa",
    "tag": "tag",
    "prefixo": "prefixo",
    "marca": "marca",
    "categ": "categoria",
    "data": "data",
    "hora": "hora",
    "entrada": "entrada",
    "saida": "saida",
    "permanencia": "permanencia",
    "rodovia": "rodovia",
    "praca": "praca",
    "nome": "nome_local",
    "estabelecimentos": "estabelecimento",
    "descricao": "descricao",
    "referencia": "referencia",
    "inicio": "inicio",
    "viagem": "viagem",
    "embarcador": "embarcador",
    "cnpj": "cnpj",
    "valor": "valor",
}

base_eventos = []
abas_eventos = [a for a in xls.sheet_names if normalizar_texto(a) in MAPA_TIPO]

for aba in abas_eventos:
    df = pd.read_excel(ARQUIVO, sheet_name=aba)
    df = normalizar_colunas(df)
    df = df.dropna(how="all")
    ren = {c: MAPA_COLUNAS[c] for c in df.columns if c in MAPA_COLUNAS}
    df = df.rename(columns=ren)
    df["origem_aba"] = aba
    df["tipo_evento"] = MAPA_TIPO[normalizar_texto(aba)]
    if "valor" in df.columns:
        df["valor"] = df["valor"].map(parse_valor)
    if "data" in df.columns:
        df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
    if "hora" in df.columns:
        df["hora"] = pd.to_datetime(df["hora"], format="%H:%M:%S", errors="coerce").dt.time
    if "entrada" in df.columns:
        df["entrada"] = pd.to_datetime(df["entrada"], dayfirst=True, errors="coerce")
    if "saida" in df.columns:
        df["saida"] = pd.to_datetime(df["saida"], dayfirst=True, errors="coerce")
    base_eventos.append(df)

df_eventos = pd.concat(base_eventos, ignore_index=True, sort=False)

# Colunas canônicas para facilitar joins e analise
colunas_canonicas = [
    "tipo_evento", "origem_aba", "placa", "tag", "prefixo", "marca", "categoria",
    "data", "hora", "entrada", "saida", "permanencia", "rodovia", "praca", "estabelecimento", "nome_local",
    "descricao", "referencia", "inicio", "viagem", "embarcador", "cnpj", "valor"
]
for c in colunas_canonicas:
    if c not in df_eventos.columns:
        df_eventos[c] = pd.NA

df_eventos = df_eventos[colunas_canonicas + [c for c in df_eventos.columns if c not in colunas_canonicas]]
df_eventos.head()

,tipo_evento,origem_aba,placa,tag,prefixo,marca,categoria,data,hora,entrada,saida,permanencia,rodovia,praca,estabelecimento,nome_local,descricao,referencia,inicio,viagem,embarcador,cnpj,valor
0,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:54:17,NaT,NaT,NaN,SP VIAS RODOVIAS INTEGRADAS DO OESTE S/A,"SP280, KM208+400, OESTE, ITATINGA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.4
1,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:26:17,NaT,NaT,NaN,SP VIAS RODOVIAS INTEGRADAS DO OESTE S/A,"SP280, KM158+300, OESTE, QUADRA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.4
2,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:01:23,NaT,NaT,NaN,VIA COLINAS,"SP280, KM 111+300, BOITUVA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.8
3,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,18:41:54,NaT,NaT,NaN,ECOVIAS RAPOSO-CASTELO,"SP280, KM33+000, OESTE, ITAPEVI",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.93
4,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,18:31:51,NaT,NaT,NaN,ECOVIAS RAPOSO-CASTELO,"SP280, KM18+000, OESTE, OSASCO",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.8


In [4]:
# Resumo da fatura em formato tidy (chave-valor + blocos de totais)
df_resumo = pd.read_excel(ARQUIVO, sheet_name="RESUMO DA FATURA")
df_resumo = normalizar_colunas(df_resumo)

cabecalho = (
    df_resumo.loc[:5, ["nome", "descricao"]]
    .dropna(how="all")
    .rename(columns={"nome": "campo", "descricao": "valor"})
)
cabecalho["campo"] = cabecalho["campo"].map(normalizar_texto)

totais = df_resumo[[c for c in ["nome", "descricao", "qtde", "valor"] if c in df_resumo.columns]].copy()
totais = totais.dropna(how="all")
if "valor" in totais.columns:
    totais["valor"] = totais["valor"].map(parse_valor)

cabecalho, totais.head(20)

(        campo                                              valor
 0      numero                                         2658904195
 1        nome  RSR COMERCIO DE FERRO AÇO E LOCAÇAO DE EQUIP LTDA
 2     emissao                                         08/03/2026
 3  vencimento                                         15/03/2026
 4       valor                                             517.63,
                                                  nome                                          descricao
 0                                              Número                                         2658904195
 1                                                Nome  RSR COMERCIO DE FERRO AÇO E LOCAÇAO DE EQUIP LTDA
 2                                             Emissão                                         08/03/2026
 3                                          Vencimento                                         15/03/2026
 4                                               Valor                   

In [7]:
# Visoes de apoio para analise rapida
# display(df_eventos.groupby("tipo_evento", dropna=False)["valor"].agg(["count", "sum"]).sort_values("sum", ascending=False))
# display(df_eventos.groupby(["placa", "tipo_evento"], dropna=False)["valor"].sum().reset_index().sort_values("valor", ascending=False).head(20))

# Se quiser persistir a tabela normalizada:
# df_eventos.to_parquet("../02-Referencias/relatorio_20260331024626_normalizado.parquet", index=False)
# df_eventos.to_csv("../02-Referencias/relatorio_20260331024626_normalizado.csv", index=False, sep=';')
df_eventos.head()

,tipo_evento,origem_aba,placa,tag,prefixo,marca,categoria,data,hora,entrada,saida,permanencia,rodovia,praca,estabelecimento,nome_local,descricao,referencia,inicio,viagem,embarcador,cnpj,valor
0,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:54:17,NaT,NaT,NaN,SP VIAS RODOVIAS INTEGRADAS DO OESTE S/A,"SP280, KM208+400, OESTE, ITATINGA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.4
1,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:26:17,NaT,NaT,NaN,SP VIAS RODOVIAS INTEGRADAS DO OESTE S/A,"SP280, KM158+300, OESTE, QUADRA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.4
2,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,20:01:23,NaT,NaT,NaN,VIA COLINAS,"SP280, KM 111+300, BOITUVA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.8
3,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,18:41:54,NaT,NaT,NaN,ECOVIAS RAPOSO-CASTELO,"SP280, KM33+000, OESTE, ITAPEVI",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.93
4,pedagio,PASSAGENS PEDÁGIO,TKQ2D62,747920429,,VW VOLKSWAGEN,1,2026-02-08,18:31:51,NaT,NaT,NaN,ECOVIAS RAPOSO-CASTELO,"SP280, KM18+000, OESTE, OSASCO",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.8
